
# <span style="color:rgb(213,80,0)">PROF. AMBROSINO R.\_\_\_ESAME\_\_\_20/04/2026\_\_\_SALVATORE\_RAIOLA\_DE6000008</span>


In [1]:
clear; close all; clc;


## 0)  DEFINIZIONI
#### Variabili simboliche

In [2]:
syms h1 h2 h3 dh1 dh2 dh3 u1 u2 w real


#### Parametri

In [3]:
A1 = 1;
A2 = 1;
A3 = 1;

a12 = 0.6;
a23 = 0.4;
a2 = 0.3;
a3 = 0.25;


#### Modello Dinamico

In [4]:
% Ingresso
u = [u1;
     u2];

% Stato
x = [h1;
     h2;
     h3];

% Uscita
y = x;

% Equazioni del sistema
eq_1 = -A1*dh1 + u1 - a12*sqrt(h1-h2) == 0;
eq_2 = -A2*dh2 + a12*sqrt(h1-h2) - a23*sqrt(h2-h3) - a2*sqrt(h2) == 0;
eq_3 = -A3*dh3 + u2 + a23*sqrt(h2-h3) - a3*sqrt(h3) == 0;

eqs = [eq_1;
       eq_2;
       eq_3];

sol = solve(eqs, [dh1, dh2, dh3], "ReturnConditions",true);

% dx/dt
f = [sol.dh1;
     sol.dh2;
     sol.dh3];
% y
g = x;


## 1)  PUNTI DI EQUILIBRIO E LINEARIZZAZIONE


In [5]:
eq_dh1_sym = sol.dh1;
eq_dh2_sym = sol.dh2;
eq_dh3_sym = sol.dh3;

x_star_1 = 3.274;
x_star_2 = 2;
x_star_3 = 1.6;

% Punto di Equilibrio
x_star = [x_star_1;
          x_star_2;
          x_star_3];


### 1.1)  Valori di u1\* e u2\* che rendono x\* un punto di equilibrio

In [6]:

eq_dh1 = subs(eq_dh1_sym, {h1, h2, h3}, {x_star_1, x_star_2, x_star_3}) == 0;
eq_dh3 = subs(eq_dh3_sym, {h1, h2, h3}, {x_star_1, x_star_2, x_star_3}) == 0;

u_star_sol = solve([eq_dh1, eq_dh3], [u1, u2], 'ReturnConditions',true);

u1_star = double(u_star_sol.u1);
u2_star = double(u_star_sol.u2);

disp(table(x_star_1, x_star_2, x_star_3, u1_star, u2_star));

    x_star_1    x_star_2    x_star_3    u1_star    u2_star 
    ________    ________    ________    _______    ________
     3.274         2          1.6       0.67723    0.063246



### 1.2)  Modello Linearizzato nell'intorno di (x\*, u\*)

In [7]:
u_star = [u1_star;
          u2_star];

A_sym = jacobian(f, x);
B_sym = jacobian(f, u);
C_sym = jacobian(g, x);
D_sym = jacobian(g, u);


### 1.3)  Matrici del Sistema

In [8]:
A = double(subs(A_sym, [x; u], [x_star; u_star]));
B = double(subs(B_sym, [x; u], [x_star; u_star]));
C = double(subs(C_sym, [x; u], [x_star; u_star]));
D = double(subs(D_sym, [x; u], [x_star; u_star]));
n = size(A,1);
m = size(B,2);

sys = ss(A, B, C, D)

sys =
 
  A = 
            x1       x2       x3
   x1  -0.2658   0.2658        0
   x2   0.2658  -0.6881   0.3162
   x3        0   0.3162   -0.415
 
  B = 
       u1  u2
   x1   1   0
   x2   0   0
   x3   0   1
 
  C = 
       x1  x2  x3
   y1   1   0   0
   y2   0   1   0
   y3   0   0   1
 
  D = 
       u1  u2
   y1   0   0
   y2   0   0
   y3   0   0
 
Continuous-time state-space model.
Model Properties



### 1.4) Verifica Stabilità x\*, raggiungibilità e osservabilità sistema
#### Stabilità

In [9]:
% Approccio agli autovalori

autovalori_A = eig(A);

if (all(autovalori_A < 0))
    disp('Il punto di equiibrio è asintoticamente stabile'); autovalori_A
elseif (any(autovalori_A == 0))
    disp('Il punto di equilibrio è stabile'); autovalori_A
else
    disp('Il punto di equilibrio è instabile'); autovalori_A
end

Il punto di equiibrio è asintoticamente stabile
autovalori_A = 3x1
   -0.9690
   -0.3390
   -0.0609


#### Raggiungibilità

In [10]:
if (rank(ctrb(A,B)) == n)
    disp('Il sistema è completamente raggiungibile');
else
    disp('Il sistema NON è completamente raggiungibile');
end

Il sistema è completamente raggiungibile


#### Osservabilità

In [11]:
if (rank(obsv(A,C)) == n)
    disp('Il sistema è completamente osservabile');
else
    disp('Il sistema NON è completamente osservabile');
end

Il sistema è completamente osservabile



## 2)  CONTROLLO STABILIZZANTE MEDIANTE APPROCCIO ALLA LYAPUNOV
#### Specifiche

In [12]:
% Stato iniziale
h0 = [0.3;
      0.2;
      0.1];

C_sel = C(1:2,:);

% Riferimento
h1_ref_lmi = 0.5;
h2_ref_lmi = 0.4;

h_ref = [h1_ref_lmi;
         h2_ref_lmi];


#### Risoluzione con cvx

In [13]:
cvx_clear;
cvx_begin sdp quiet
    cvx_precision high
    variable Q(n,n) symmetric
    variable L(m,n)
    minimize(0);
    subject to
        Q >= (1e-6)*eye(n);
        A*Q + Q*A' + B*L + L'*B' <= (1e-6)*eye(n);
cvx_end;

K_LMI = -L/Q

K_LMI = 2x3
    0.2260    0.5389   -0.0185
   -0.0011    0.6404    0.0733


#### Compensazione del Guadagno

In [14]:
gain = dcgain(ss((A-B*K_LMI), B, C_sel, D(1:2, 1:2)));
N = pinv(gain)

N = 2x2
    0.5073    0.2328
   -0.4115    1.3868


#### Verifica Simulazione

Schema Simulink: [**Verifica\_LMI**](matlab:open_system('Verifica_LMI.slx'))


## 3)  CONTROLLO LQR CON AZIONE INTEGRALE
#### Specifiche

In [15]:
% Riferimento
h2_ref_lqr = 2.8;

% Tempo di Assestamento
Ts = 1;


#### Sistema Aumentato

In [16]:
C_LQR = [0, 1, 0];

p_a = size(C_LQR, 1);

A_a = [A, zeros(n, p_a);
       -C_LQR, 0]

A_a = 4x4
   -0.2658    0.2658         0         0
    0.2658   -0.6881    0.3162         0
         0    0.3162   -0.4150         0
         0   -1.0000         0         0

In [17]:
n_a = size(A_a, 1);

B_a = [B;
       zeros(p_a,m)]

B_a = 4x2
     1     0
     0     0
     0     1
     0     0


#### Sintesi LQR

In [18]:
Q_x = eye(n);
Q_z = 10*eye(p_a);

Q = blkdiag(Q_x, Q_z);

R = eye(m);

% Trasliamo lo spettro di A_a per rispettare la specifica sul Ts
alpha = 4/Ts;
A_hat = A_a + alpha*eye(n_a);

[K_LQR, ~, ~] = lqr(A_hat, B_a, Q, R)

K_LQR = 2x4
   13.5851  251.7205    7.1912 -706.4912
    7.1912  301.5251   15.9478 -841.6954

In [19]:
K_LQR_x = K_LQR(:, 1:n)

K_LQR_x = 2x3
   13.5851  251.7205    7.1912
    7.1912  301.5251   15.9478

In [20]:
K_LQR_i = K_LQR(:, n+1:end)

K_LQR_i = 2x1
 -706.4912
 -841.6954


#### Verifica Simulazione

Schema Simulink: [**Verifica\_LQR**](matlab:open_system('Verifica_LQR.slx'))


## **4)  CONTROLLO H\-INF**
#### Sistema con Disturbo

In [21]:
% Equazione con disturbo
eq_2 = -A2*dh2 + a12*sqrt(h1-h2) - a23*sqrt(h2-h3) - a2*sqrt(h2) + w == 0;

eqs_w = [eq_1;
         eq_2;
         eq_3];

sol_w = solve(eqs_w, [dh1, dh2, dh3], "ReturnConditions",true);

% dx/dt
f_w = [sol_w.dh1;
       sol_w.dh2;
       sol_w.dh3];

u_w = [u;
       w];

ro = 0.1;

% y
g_w = [x;
      ro*u];

A_w_sym = jacobian(f_w, x);
B_u_sym = jacobian(f_w, u);
B_w_sym = jacobian(f_w, w);
C_w_sym = jacobian(g_w, x);
D_w_sym = jacobian(g_w, u_w);

B_tot_sym = jacobian(f_w, u_w);

A_w = double(subs(A_w_sym, [x; u], [x_star; u_star]));
B_u = double(subs(B_u_sym, [x; u], [x_star; u_star]));
B_w = double(subs(B_w_sym, [x; u], [x_star; u_star]));
C_w = double(subs(C_w_sym, [x; u], [x_star; u_star]));
D_w = double(subs(D_w_sym, [x; u], [x_star; u_star]));

B_tot = double(subs(B_tot_sym, [x; u], [x_star; u_star]));

sys_w = ss(A_w, B_tot, C_w, D_w)

sys_w =
 
  A = 
            x1       x2       x3
   x1  -0.2658   0.2658        0
   x2   0.2658  -0.6881   0.3162
   x3        0   0.3162   -0.415
 
  B = 
       u1  u2  u3
   x1   1   0   0
   x2   0   0   1
   x3   0   1   0
 
  C = 
       x1  x2  x3
   y1   1   0   0
   y2   0   1   0
   y3   0   0   1
   y4   0   0   0
   y5   0   0   0
 
  D = 
        u1   u2   u3
   y1    0    0    0
   y2    0    0    0
   y3    0    0    0
   y4  0.1    0    0
   y5    0  0.1    0
 
Continuous-time state-space model.
Model Properties

In [22]:
B_w

B_w = 3x1
     0
     1
     0


#### Sintesi H\-inf

In [23]:
[K_HINF, ~, gamma] = hinfsyn(sys_w, 5, 2);
K_HINF

K_HINF =
 
  A = 
               x1          x2          x3
   x1     -0.2658      0.2658  -2.704e-25
   x2      0.2499     -0.7042      0.2809
   x3    -0.01925      0.2957      -0.462
 
  B = 
               u1          u2          u3          u4          u5
   x1   6.057e-27  -1.041e-25     6.2e-26          10   1.944e-23
   x2   4.339e-16  -5.372e-14  -1.545e-24   -3.44e-16   1.157e-23
   x3   5.687e-15   -2.08e-13   8.473e-13   6.427e-17   3.477e-24
 
  C = 
             x1        x2        x3
   y1  -0.01925  -0.02053  -0.04691
   y2   -0.0159  -0.01609  -0.03537
 
  D = 
       u1  u2  u3  u4  u5
   y1   0   0   0   0   0
   y2   0   0   0   0   0
 
Continuous-time state-space model.
Model Properties

In [24]:
% Controllo
if gamma>1
    disp('MIXSYN  FAILED:');
    double(gamma)
else
    fprintf("\n---- MIXSYN RESULT ----\n");
    double(gamma)
end

---- MIXSYN RESULT ----
ans = 0